# 🏠 House Prices - Evaluace a Porovnání Modelů

Tento notebook obsahuje **evaluaci a porovnání** všech natrénovaných modelů.

## 📋 Struktura:

1. **Načtení dat a modelů** z `models/`
2. **EXPERIMENT 1:** Evaluace Baseline Linear Models
3. **EXPERIMENT 2:** Evaluace Tree-based Models
4. **EXPERIMENT 3:** Evaluace Tuned Models
5. **POROVNÁNÍ VŠECH EXPERIMENTŮ**
6. **FINÁLNÍ MODEL** a predikce

---

## 🎯 Evaluace zahrnuje:

- **Cross-validation** pro robustní evaluaci
- **Metriky:** RMSE, MAE, R²
- **Vizualizace** výsledků
- **Porovnání** všech modelů

---

## ⚠️ DŮLEŽITÉ:

- Modely se načítají z `models/` (natrénované v `03_Modeling.ipynb`)
- Všechny experimenty používají **stejnou CV strategii** pro objektivní porovnání
- Finální model se vybere na základě CV RMSE


## 1. Načtení dat a modelů


In [ ]:
# 📦 Import knihoven
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ML metriky
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 📍 Nastavení cest
PROJECT_DIR = Path('..')
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
MODELS_DIR = PROJECT_DIR / 'models'
RESULTS_DIR.mkdir(exist_ok=True)

# Nastavení pro reprodukovatelnost
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Všechny knihovny načteny!")


In [ ]:
# Načtení preprocessovaných dat
with open(DATA_DIR / 'processed_data.pkl', 'rb') as f:
    processed_data = pickle.load(f)
    X_train = processed_data['X_train']
    X_test = processed_data['X_test']
    y_train = processed_data['y_train']
    test_ids = processed_data['test_ids']

print("=" * 80)
print("📊 NAČTENÁ DATA")
print("=" * 80)
print(f"   X_train: {X_train.shape}")
print(f"   X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape} (log SalePrice)")
print(f"   Features: {X_train.shape[1]}")
print("\n✅ Data připravena pro evaluaci!")


## 2. EXPERIMENT 1: Evaluace Baseline Linear Models


In [ ]:
# Načtení modelů a konfigurace
with open(MODELS_DIR / 'exp1_config.pkl', 'rb') as f:
    EXP1_CONFIG = pickle.load(f)

exp1_models = {}
for model_name in ['Ridge', 'Lasso']:
    with open(MODELS_DIR / f'exp1_{model_name.lower()}.pkl', 'rb') as f:
        exp1_models[model_name] = pickle.load(f)

print("=" * 80)
print(f"🔬 {EXP1_CONFIG['name']} - EVALUACE")
print("=" * 80)
print(f"   CV folds: {EXP1_CONFIG['cv_folds']}")
print(f"   Modely: {', '.join(exp1_models.keys())}")


In [ ]:
# Cross-validation pro každý model
print("=" * 80)
print("🔄 CROSS-VALIDATION")
print("=" * 80)

kfold = KFold(n_splits=EXP1_CONFIG['cv_folds'], shuffle=True, random_state=RANDOM_STATE)
exp1_results = {}

for name, model in exp1_models.items():
    # CV skóre (RMSE)
    cv_scores = cross_val_score(
        model, X_train, y_train,
        scoring='neg_root_mean_squared_error',
        cv=kfold,
        n_jobs=-1
    )
    cv_rmse = -cv_scores  # Convert to positive
    
    # Train predikce pro srovnání
    train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)
    train_r2 = r2_score(y_train, train_pred)
    
    # Uložení výsledků
    exp1_results[name] = {
        'cv_rmse_mean': cv_rmse.mean(),
        'cv_rmse_std': cv_rmse.std(),
        'train_rmse': train_rmse,
        'train_mae': train_mae,
        'train_r2': train_r2,
        'overfitting': train_rmse - cv_rmse.mean()
    }
    
    print(f"\n📈 {name}:")
    print(f"   CV RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")
    print(f"   Train RMSE: {train_rmse:.4f}")
    print(f"   Train MAE: {train_mae:.4f}")
    print(f"   Train R²: {train_r2:.4f}")
    print(f"   Overfitting: {exp1_results[name]['overfitting']:.4f}")


In [ ]:
# Vizualizace výsledků
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for idx, (name, model) in enumerate(exp1_models.items()):
    y_pred = model.predict(X_train)
    
    axes[idx].scatter(y_train, y_pred, alpha=0.5)
    min_val = min(y_train.min(), y_pred.min())
    max_val = max(y_train.max(), y_pred.max())
    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
    axes[idx].set_xlabel('Actual (log SalePrice)', fontsize=12)
    axes[idx].set_ylabel('Predicted (log SalePrice)', fontsize=12)
    axes[idx].set_title(f'{name} - Predictions vs Actual\nCV RMSE: {exp1_results[name]["cv_rmse_mean"]:.4f}', fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'exp1_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Grafy uloženy: results/exp1_predictions.png")


In [ ]:
# Závěr experimentu 1
exp1_df = pd.DataFrame(exp1_results).T
exp1_df = exp1_df.sort_values('cv_rmse_mean')

print("=" * 80)
print("📊 POROVNÁNÍ EXPERIMENT 1")
print("=" * 80)
print(exp1_df[['cv_rmse_mean', 'cv_rmse_std', 'train_rmse', 'train_r2', 'overfitting']].round(4))

best_exp1 = exp1_df.index[0]
print(f"\n🏆 Nejlepší model: {best_exp1}")
print(f"   CV RMSE: {exp1_df.loc[best_exp1, 'cv_rmse_mean']:.4f}")

# Uložení výsledků
exp1_df.to_csv(RESULTS_DIR / 'exp1_results.csv')
print(f"\n✅ Výsledky uloženy: results/exp1_results.csv")


## 3. EXPERIMENT 2: Evaluace Tree-based Models


In [ ]:
# Načtení modelů a konfigurace
with open(MODELS_DIR / 'exp2_config.pkl', 'rb') as f:
    EXP2_CONFIG = pickle.load(f)

exp2_models = {}
for model_name in ['RandomForest', 'XGBoost', 'LightGBM']:
    with open(MODELS_DIR / f'exp2_{model_name.lower()}.pkl', 'rb') as f:
        exp2_models[model_name] = pickle.load(f)

print("=" * 80)
print(f"🔬 {EXP2_CONFIG['name']} - EVALUACE")
print("=" * 80)
print(f"   CV folds: {EXP2_CONFIG['cv_folds']}")
print(f"   Modely: {', '.join(exp2_models.keys())}")


In [ ]:
# Cross-validation pro každý model
print("=" * 80)
print("🔄 CROSS-VALIDATION")
print("=" * 80)

kfold = KFold(n_splits=EXP2_CONFIG['cv_folds'], shuffle=True, random_state=RANDOM_STATE)
exp2_results = {}

for name, model in exp2_models.items():
    print(f"\n📊 {name}...")
    cv_scores = cross_val_score(
        model, X_train, y_train,
        scoring='neg_root_mean_squared_error',
        cv=kfold,
        n_jobs=-1
    )
    cv_rmse = -cv_scores
    
    train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)
    train_r2 = r2_score(y_train, train_pred)
    
    exp2_results[name] = {
        'cv_rmse_mean': cv_rmse.mean(),
        'cv_rmse_std': cv_rmse.std(),
        'train_rmse': train_rmse,
        'train_mae': train_mae,
        'train_r2': train_r2,
        'overfitting': train_rmse - cv_rmse.mean()
    }
    
    print(f"   CV RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")
    print(f"   Train RMSE: {train_rmse:.4f}")
    print(f"   Train R²: {train_r2:.4f}")
    print(f"   Overfitting: {exp2_results[name]['overfitting']:.4f}")


In [ ]:
# Feature importance pro tree modely
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, model) in enumerate(exp2_models.items()):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1][:15]
        
        axes[idx].barh(range(15), importances[indices])
        axes[idx].set_yticks(range(15))
        axes[idx].set_yticklabels([X_train.columns[i] for i in indices], fontsize=8)
        axes[idx].set_xlabel('Feature Importance', fontsize=10)
        axes[idx].set_title(f'{name} - Top 15 Features', fontsize=12)
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'exp2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Grafy uloženy: results/exp2_feature_importance.png")


In [ ]:
# Závěr experimentu 2
exp2_df = pd.DataFrame(exp2_results).T
exp2_df = exp2_df.sort_values('cv_rmse_mean')

print("=" * 80)
print("📊 POROVNÁNÍ EXPERIMENT 2")
print("=" * 80)
print(exp2_df[['cv_rmse_mean', 'cv_rmse_std', 'train_rmse', 'train_r2', 'overfitting']].round(4))

best_exp2 = exp2_df.index[0]
print(f"\n🏆 Nejlepší model: {best_exp2}")
print(f"   CV RMSE: {exp2_df.loc[best_exp2, 'cv_rmse_mean']:.4f}")

exp2_df.to_csv(RESULTS_DIR / 'exp2_results.csv')
print(f"\n✅ Výsledky uloženy: results/exp2_results.csv")


## 4. EXPERIMENT 3: Evaluace Tuned Models


In [ ]:
# Načtení modelu a konfigurace
with open(MODELS_DIR / 'exp3_config.pkl', 'rb') as f:
    EXP3_CONFIG = pickle.load(f)

base_model_name = EXP3_CONFIG['base_model']
with open(MODELS_DIR / f'exp3_{base_model_name.lower()}_tuned.pkl', 'rb') as f:
    exp3_model = pickle.load(f)

print("=" * 80)
print(f"🔬 {EXP3_CONFIG['name']} - EVALUACE")
print("=" * 80)
print(f"   Base model: {base_model_name}")
print(f"   Best parameters: {EXP3_CONFIG['best_params']}")


In [ ]:
# Cross-validation
print("=" * 80)
print("🔄 CROSS-VALIDATION")
print("=" * 80)

kfold = KFold(n_splits=EXP3_CONFIG['cv_folds'], shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    exp3_model, X_train, y_train,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    n_jobs=-1
)
cv_rmse = -cv_scores

train_pred = exp3_model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_mae = mean_absolute_error(y_train, train_pred)
train_r2 = r2_score(y_train, train_pred)

exp3_results = {
    'cv_rmse_mean': cv_rmse.mean(),
    'cv_rmse_std': cv_rmse.std(),
    'train_rmse': train_rmse,
    'train_mae': train_mae,
    'train_r2': train_r2,
    'overfitting': train_rmse - cv_rmse.mean()
}

print(f"\n📈 {base_model_name}_tuned:")
print(f"   CV RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})")
print(f"   Train RMSE: {train_rmse:.4f}")
print(f"   Train R²: {train_r2:.4f}")
print(f"   Overfitting: {exp3_results['overfitting']:.4f}")

# Porovnání před a po tuning
if base_model_name in exp2_results:
    print("\n" + "=" * 80)
    print("📊 POROVNÁNÍ: Před vs Po Tuning")
    print("=" * 80)
    print(f"{'Metric':<20} {'Před tuning':<15} {'Po tuning':<15} {'Zlepšení':<15}")
    print("-" * 80)
    print(f"{'CV RMSE':<20} {exp2_results[base_model_name]['cv_rmse_mean']:<15.4f} {exp3_results['cv_rmse_mean']:<15.4f} {exp2_results[base_model_name]['cv_rmse_mean'] - exp3_results['cv_rmse_mean']:<15.4f}")
    print(f"{'Train RMSE':<20} {exp2_results[base_model_name]['train_rmse']:<15.4f} {train_rmse:<15.4f} {exp2_results[base_model_name]['train_rmse'] - train_rmse:<15.4f}")
    print(f"{'Train R²':<20} {exp2_results[base_model_name]['train_r2']:<15.4f} {train_r2:<15.4f} {train_r2 - exp2_results[base_model_name]['train_r2']:<15.4f}")

exp3_df = pd.DataFrame([exp3_results], index=[f'{base_model_name}_tuned'])
exp3_df.to_csv(RESULTS_DIR / 'exp3_results.csv')
print(f"\n✅ Výsledky uloženy: results/exp3_results.csv")


## 5. POROVNÁNÍ VŠECH EXPERIMENTŮ


In [ ]:
# Sestavení comparison table
all_results = []

# Experiment 1
for name, results in exp1_results.items():
    all_results.append({
        'Experiment': 'EXP_001',
        'Model': name,
        'CV_RMSE_mean': results['cv_rmse_mean'],
        'CV_RMSE_std': results['cv_rmse_std'],
        'Train_RMSE': results['train_rmse'],
        'Train_R2': results['train_r2'],
        'Overfitting': results['overfitting']
    })

# Experiment 2
for name, results in exp2_results.items():
    all_results.append({
        'Experiment': 'EXP_002',
        'Model': name,
        'CV_RMSE_mean': results['cv_rmse_mean'],
        'CV_RMSE_std': results['cv_rmse_std'],
        'Train_RMSE': results['train_rmse'],
        'Train_R2': results['train_r2'],
        'Overfitting': results['overfitting']
    })

# Experiment 3
all_results.append({
    'Experiment': 'EXP_003',
    'Model': f'{base_model_name}_tuned',
    'CV_RMSE_mean': exp3_results['cv_rmse_mean'],
    'CV_RMSE_std': exp3_results['cv_rmse_std'],
    'Train_RMSE': exp3_results['train_rmse'],
    'Train_R2': exp3_results['train_r2'],
    'Overfitting': exp3_results['overfitting']
})

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df.sort_values('CV_RMSE_mean')

print("=" * 80)
print("📊 POROVNÁNÍ VŠECH EXPERIMENTŮ")
print("=" * 80)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(RESULTS_DIR / 'all_experiments_comparison.csv', index=False)
print(f"\n✅ Porovnání uloženo: results/all_experiments_comparison.csv")


In [ ]:
# Vizualizace porovnání
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# CV RMSE porovnání
axes[0].barh(range(len(comparison_df)), comparison_df['CV_RMSE_mean'], 
             xerr=comparison_df['CV_RMSE_std'], capsize=5)
axes[0].set_yticks(range(len(comparison_df)))
axes[0].set_yticklabels(comparison_df['Model'], fontsize=10)
axes[0].set_xlabel('CV RMSE', fontsize=12)
axes[0].set_title('Cross-Validation RMSE Comparison', fontsize=14)
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Train vs CV RMSE
x_pos = np.arange(len(comparison_df))
width = 0.35
axes[1].barh(x_pos - width/2, comparison_df['CV_RMSE_mean'], width, 
             label='CV RMSE', alpha=0.8)
axes[1].barh(x_pos + width/2, comparison_df['Train_RMSE'], width, 
             label='Train RMSE', alpha=0.8)
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(comparison_df['Model'], fontsize=10)
axes[1].set_xlabel('RMSE', fontsize=12)
axes[1].set_title('Train vs CV RMSE', fontsize=14)
axes[1].legend()
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'all_experiments_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Grafy uloženy: results/all_experiments_comparison.png")
